In [1]:
import pandas as pd
import numpy as np

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
df = pd.read_csv('/content/drive/MyDrive/26봄1차인콘/df_final.csv')

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4639120 entries, 0 to 4639119
Data columns (total 32 columns):
 #   Column                                     Dtype  
---  ------                                     -----  
 0   ACCOUNT_ID                                 object 
 1   CARD_ID                                    object 
 2   TRANSACTION_ID                             object 
 3   GROSS_TRANSACTION_AMOUNT                   float64
 4   TRANSACTION_DATE                           object 
 5   TRANSACTION_TYPE                           object 
 6   TRANSACTION_STATE                          object 
 7   TRANSACTION_CITY                           object 
 8   MERCHANT_STATE                             object 
 9   MERCHANT_CITY                              object 
 10  MERCHANT_CATEGORY_LEVEL_1                  object 
 11  MERCHANT_CATEGORY_LEVEL_2                  object 
 12  MERCHANT_CATEGORY_LEVEL_3                  object 
 13  TRANSACTION_DESCRIPTION                   

In [5]:
# 계층 구조를 트리 형태로 출력 (중복 제거)
hierarchy = df[['MERCHANT_CATEGORY_LEVEL_1', 'MERCHANT_CATEGORY_LEVEL_2', 'MERCHANT_CATEGORY_LEVEL_3']].drop_duplicates()

# Level 1별로 그룹화해서 보기
for l1 in hierarchy['MERCHANT_CATEGORY_LEVEL_1'].unique():
    print(f"\n[Level 1] {l1}")
    l2_list = hierarchy[hierarchy['MERCHANT_CATEGORY_LEVEL_1'] == l1]['MERCHANT_CATEGORY_LEVEL_2'].unique()

    for l2 in l2_list:
        print(f"  └── [Level 2] {l2}")
        l3_list = hierarchy[(hierarchy['MERCHANT_CATEGORY_LEVEL_1'] == l1) &
                            (hierarchy['MERCHANT_CATEGORY_LEVEL_2'] == l2)]['MERCHANT_CATEGORY_LEVEL_3'].unique()
        for l3 in l3_list:
            print(f"      ├── [Level 3] {l3}")


[Level 1] Restaurants and Food Services
  └── [Level 2] QSR
      ├── [Level 3] QSR Burgers
      ├── [Level 3] QSR Sandwiches
      ├── [Level 3] QSR Chicken
      ├── [Level 3] Coffee / Tea
      ├── [Level 3] QSR Smoothies
      ├── [Level 3] Dessert
      ├── [Level 3] Pizza Restaurants
      ├── [Level 3] Mexican Restaurants
      ├── [Level 3] QSR Healthy
  └── [Level 2] Casual Dining
      ├── [Level 3] American Restaurants
      ├── [Level 3] Breakfast Restaurants
      ├── [Level 3] Pizza Restaurants
      ├── [Level 3] Mexican Restaurants
      ├── [Level 3] Asian Restaurants
      ├── [Level 3] Seafood Restaurants
      ├── [Level 3] Carribean
      ├── [Level 3] Italian Restaurants
      ├── [Level 3] Mediterranean Restaurants
  └── [Level 2] Vending & Beverage Retailers
      ├── [Level 3] Vending & Beverage Retailers
  └── [Level 2] Food Services
      ├── [Level 3] Food Services
  └── [Level 2] Food Delivery
      ├── [Level 3] Food Delivery

[Level 1] Consumer Services

In [6]:
import pandas as pd

# 1. 미국 주(State)를 4대 지역(Region)으로 매핑 (미국 인구조사국 기준)
state_to_region = {
    # 동북부 (Northeast)
    'CT': 'Northeast', 'ME': 'Northeast', 'MA': 'Northeast', 'NH': 'Northeast', 'RI': 'Northeast', 'VT': 'Northeast', 'NJ': 'Northeast', 'NY': 'Northeast', 'PA': 'Northeast',
    # 중서부 (Midwest)
    'IL': 'Midwest', 'IN': 'Midwest', 'MI': 'Midwest', 'OH': 'Midwest', 'WI': 'Midwest', 'IA': 'Midwest', 'KS': 'Midwest', 'MN': 'Midwest', 'MO': 'Midwest', 'NE': 'Midwest', 'ND': 'Midwest', 'SD': 'Midwest',
    # 남부 (South)
    'DE': 'South', 'FL': 'South', 'GA': 'South', 'MD': 'South', 'NC': 'South', 'SC': 'South', 'VA': 'South', 'DC': 'South', 'WV': 'South', 'AL': 'South', 'KY': 'South', 'MS': 'South', 'TN': 'South', 'AR': 'South', 'LA': 'South', 'OK': 'South', 'TX': 'South',
    # 서부 (West)
    'AZ': 'West', 'CO': 'West', 'ID': 'West', 'MT': 'West', 'NV': 'West', 'NM': 'West', 'UT': 'West', 'WY': 'West', 'AK': 'West', 'CA': 'West', 'HI': 'West', 'OR': 'West', 'WA': 'West'
}


In [7]:
# 1. 지역(Region) 칼럼 새로 만들기
df['REGION'] = df['CARD_HOLDER_STATE'].map(state_to_region)

# 2. 지역별/결제방식별/음식종류별로 묶어서 숫자 세기
# 여기서 'ORDER_TYPE'은 배달/현장결제를 구분하는 칼럼명이라고 가정합시다.
analysis_base = df.groupby(['REGION', 'CARD_PRESENT_INDICATOR', 'MERCHANT_CATEGORY_LEVEL_3']).size().reset_index(name='count')

# 3. 비중(Local Ratio) 계산하기
# 각 지역 내에서 해당 조합이 차지하는 비율을 구합니다.

In [10]:
# print("\n=== 핵심 컬럼 유니크 값 수 ===")
cols = ['REGION', 'CARD_PRESENT_INDICATOR',
         'MERCHANT_CATEGORY_LEVEL_2', 'MERCHANT_CATEGORY_LEVEL_3', 'MERCHANT_NAME']
for col in cols:
  print(f"{col}: {df[col].nunique()}개 (결측: {df[col].isna().sum():,}개)")

REGION: 4개 (결측: 22,488개)
CARD_PRESENT_INDICATOR: 3개 (결측: 0개)
MERCHANT_CATEGORY_LEVEL_2: 40개 (결측: 197,771개)
MERCHANT_CATEGORY_LEVEL_3: 87개 (결측: 197,796개)
MERCHANT_NAME: 685개 (결측: 0개)


In [12]:
# ==========================================
# 1. 기초 집계
# =========================================
# 식음료 필터링
food_mcl2 = ['QSR', 'Casual Dining', 'Vending & Beverage Retailers', 'Food Services', 'Grocery and Delivery', 'Food Delivery',
            'Gas / Convenience', 'Nutrition & Vitamin Retailers']

df_food = df[
    df['MERCHANT_CATEGORY_LEVEL_2'].isin(food_mcl2) &
    df['CARD_HOLDER_STATE'].notna() &
    df['CARD_HOLDER_GENERATION'].notna() &
    df['MERCHANT_CATEGORY_LEVEL_3'].notna()
].copy()

# 지역별, 온/오프라인별, 음식종류별 건수 계산
# group_counts = df_food.groupby(['REGION', 'CARD_PRESENT_INDICATOR', 'MERCHANT_CATEGORY_LEVEL_3']).size().reset_index(name='count')
group_counts = df_food.groupby(['REGION', 'MERCHANT_CATEGORY_LEVEL_3']).size().reset_index(name='count')

# 각 지역별로 전체 거래가 몇 건인지 계산 (비율을 구하기 위해)
region_total = df_food.groupby('REGION').size().reset_index(name='region_total')

# 두 데이터를 합칩니다
analysis_df = group_counts.merge(region_total, on='REGION')

# 해당 지역 내에서의 비중(Local Ratio) 계산
analysis_df['local_ratio'] = analysis_df['count'] / analysis_df['region_total']

# ==========================================
# 2. 전국 평균(Global Ratio) 계산 (비교 기준)
# ==========================================
# 전국적으로 [온/오프라인 x 음식종류]의 비중이 어떤지 계산합니다.
global_total = len(df_food)
# global_counts = df_food.groupby(['CARD_PRESENT_INDICATOR', 'MERCHANT_CATEGORY_LEVEL_3']).size().reset_index(name='global_count')
global_counts = df_food.groupby(['MERCHANT_CATEGORY_LEVEL_3']).size().reset_index(name='global_count')
global_counts['global_ratio'] = global_counts['global_count'] / global_total

# ==========================================
# 3. 리프트(Lift) 계산 및 결과 도출
# ==========================================
# 내 지역 데이터와 전국 평균 데이터를 합칩니다.
final_analysis = analysis_df.merge(global_counts, on=['MERCHANT_CATEGORY_LEVEL_3'])
# final_analysis = analysis_df.merge(global_counts, on=['CARD_PRESENT_INDICATOR', 'MERCHANT_CATEGORY_LEVEL_3'])

# 리프트 계산: (우리 지역 비중 / 전국 평균 비중)
final_analysis['lift'] = final_analysis['local_ratio'] / final_analysis['global_ratio']

# 신뢰도를 위해 너무 적은 건수(예: 30건 미만)는 제외하고 리프트 순으로 정렬
result = final_analysis[final_analysis['count'] >= 30].sort_values(by=['REGION', 'lift'], ascending=[True, False])

# 보기 좋게 출력
print("=== 지역별 특화된 결제 패턴 (Lift 상위 순) ===")
# 주요 칼럼만 골라서 보기
display_cols = ['REGION', 'MERCHANT_CATEGORY_LEVEL_3', 'count', 'lift']
# display_cols = ['REGION', 'CARD_PRESENT_INDICATOR', 'MERCHANT_CATEGORY_LEVEL_3', 'count', 'lift']
print(result[display_cols])

=== 지역별 특화된 결제 패턴 (Lift 상위 순) ===
       REGION      MERCHANT_CATEGORY_LEVEL_3   count      lift
10    Midwest            Italian Restaurants     452  1.747876
0     Midwest           American Restaurants   86797  1.409580
9     Midwest                        Grocery    8358  1.353958
18    Midwest                 QSR Sandwiches  106418  1.272352
21    Midwest   Vending & Beverage Retailers  201206  1.183819
12    Midwest            Mexican Restaurants  124856  1.141044
7     Midwest                  Food Services   34591  1.058987
2     Midwest          Breakfast Restaurants   59876  1.022470
15    Midwest                    QSR Burgers  413142  0.993365
14    Midwest              Pizza Restaurants   52413  0.967615
13    Midwest  Nutrition & Vitamin Retailers     635  0.949378
5     Midwest                        Dessert   31027  0.932249
19    Midwest                  QSR Smoothies    9377  0.927705
4     Midwest                   Coffee / Tea  137165  0.868298
8     Midwest        